In [26]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
df = pd.read_parquet('./data/train/1_회원정보_train.parquet')

### C vs AB 추출하여 `ID`, `기준년월`로 병합

In [25]:
def preprocess(df):
    df_cleaned = df.copy()
    obj_cols = df_cleaned.select_dtypes(include='object').columns
    for col in obj_cols:
        sample_values = df_cleaned[col].dropna().astype(str).unique()
        if all(re.fullmatch(r'\d+대', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.replace("대", "").astype(int)
        elif all(re.fullmatch(r'\d+개', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.replace("개", "").astype(int)
        elif all(re.fullmatch(r'\d+회 이상', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d+)').astype(int)
        elif all(re.fullmatch(r'\d+일 이상', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d+)').astype(int)
        elif all(re.fullmatch(r'\d{2}\.\d+만원\+', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].str.extract(r'(\d{2})').astype(int)
        elif all(re.fullmatch(r'[A-Z]', v) for v in sample_values):
            df_cleaned[col] = df_cleaned[col].astype('category').cat.codes
        else:
            df_cleaned[col] = df_cleaned[col].astype('category').cat.codes
    return df_cleaned.fillna(0)


In [27]:
def run_correlation_isC(paths, segment_df, preprocess):
    os.makedirs('./corr_output', exist_ok=True)

    # A/B/C 필터링 후 is_C 생성
    segment_abc = segment_df[segment_df['Segment'].isin(['A', 'B', 'C'])].copy()
    segment_abc['is_C'] = (segment_abc['Segment'] == 'C').astype(int)
    segment_abc['ID'] = segment_abc['ID'].astype(str)
    segment_abc['기준년월'] = segment_abc['기준년월'].astype(str)

    for path in paths:
        name = os.path.splitext(os.path.basename(path))[0]
        print(f"\n📁 {name} 파일 처리 중...")

        # 파일 읽기
        if path.endswith('.csv'):
            df_part = pd.read_csv(path, encoding='utf-8-sig')
        elif path.endswith('.parquet'):
            df_part = pd.read_parquet(path)
        else:
            print(f"[스킵] 지원되지 않는 형식: {path}")
            continue

        df_part['ID'] = df_part['ID'].astype(str)
        df_part['기준년월'] = df_part['기준년월'].astype(str)

        # 💡 1_회원정보처럼 Segment가 이미 포함된 경우 → 병합 생략
        if 'Segment' in df_part.columns:
            df_part = df_part[df_part['Segment'].isin(['A', 'B', 'C'])].copy()
            df_part['Segment'] = df_part['Segment'].astype(str)
            df_part['is_C'] = (df_part['Segment'] == 'C').astype(int)
        else:
            df_part = pd.merge(df_part, segment_abc[['ID', '기준년월', 'is_C']], 
                               on=['ID', '기준년월'], how='inner')

        # 전처리 적용
        df_part = preprocess(df_part)

        # 상관계수 계산
        try:
            corr = df_part.corr(numeric_only=True)['is_C'].drop('is_C', errors='ignore')
            corr.to_csv(f'./corr_output/{name}_corr_is_C.csv', encoding='utf-8-sig')
            print(f"상관계수 저장 완료: {name}")
        except Exception as e:
            print(f"[에러] {name}: {e}")


In [28]:
# Segment 정보 불러오기 (train 데이터 기준)
segment_df = pd.read_parquet('./data/train/1_회원정보_train.parquet')[['ID', '기준년월', 'Segment']]

# 분석 대상 파일 리스트 (train 기준)
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet',
]

# 실행
run_correlation_isC(paths, segment_df, preprocess)


📁 1_회원정보_train 파일 처리 중...
상관계수 저장 완료: 1_회원정보_train

📁 2_신용정보_train 파일 처리 중...
상관계수 저장 완료: 2_신용정보_train

📁 3_승인매출정보_train 파일 처리 중...
상관계수 저장 완료: 3_승인매출정보_train

📁 4_청구입금정보_train 파일 처리 중...
상관계수 저장 완료: 4_청구입금정보_train

📁 5_잔액정보_train 파일 처리 중...
상관계수 저장 완료: 5_잔액정보_train

📁 6_채널정보_train 파일 처리 중...
상관계수 저장 완료: 6_채널정보_train

📁 7_마케팅정보_train 파일 처리 중...
상관계수 저장 완료: 7_마케팅정보_train

📁 8_성과정보_train 파일 처리 중...
상관계수 저장 완료: 8_성과정보_train


In [29]:
import glob

# 상관계수 결과 파일들 불러오기
corr_files = sorted(glob.glob('./corr_output/*_corr_is_C.csv'))

# 파일별로 읽고 통합
df_list = []
for file in corr_files:
    source_name = os.path.basename(file).replace('_corr_is_C.csv', '')  # 파일명만 추출
    df = pd.read_csv(file, index_col=0)
    df = df.reset_index().rename(columns={'index': 'feature', df.columns[0]: 'correlation'})
    df['source'] = source_name
    df_list.append(df)

# 전체 합치기
combined_corr = pd.concat(df_list, ignore_index=True)

# 저장
combined_corr.to_csv('./corr_output/전체_is_C_상관계수_통합.csv', index=False, encoding='utf-8-sig')

print("완료: './corr_output/전체_is_C_상관계수_통합.csv'")


완료: './corr_output/전체_is_C_상관계수_통합.csv'


In [30]:
df = pd.read_csv('./corr_output/전체_is_C_상관계수_통합.csv')

In [31]:
df

,feature,correlation,source
0,기준년월,-3.742538e-14,1_회원정보_train
1,ID,6.018434e-03,1_회원정보_train
2,남녀구분코드,1.400640e-02,1_회원정보_train
3,연령,-3.598251e-02,1_회원정보_train
4,Segment,9.841903e-01,1_회원정보_train
...,...,...,...
867,변동률_잔액_B1M,1.711321e-02,8_성과정보_train
868,변동률_잔액_일시불_B1M,4.034841e-03,8_성과정보_train
869,변동률_잔액_CA_B1M,6.253987e-03,8_성과정보_train
870,혜택수혜율_R3M,-4.328916e-04,8_성과정보_train


In [37]:
corr_df = df[df['correlation'].abs() >= 0.1]
corr_df

,feature,correlation,source
4,Segment,0.984190,1_회원정보_train
44,이용금액_R3M_신용,-0.102762,1_회원정보_train
48,_1순위카드이용금액,-0.140124,1_회원정보_train
81,카드이용한도금액,-0.140910,2_신용정보_train
82,CA한도금액,-0.129304,2_신용정보_train
107,카드이용한도금액_B1M,-0.142525,2_신용정보_train
108,카드이용한도금액_B2M,-0.143877,2_신용정보_train
167,이용금액_일시불_R12M,-0.126456,3_승인매출정보_train
168,이용금액_할부_R12M,-0.121260,3_승인매출정보_train
170,이용금액_할부_무이자_R12M,-0.142076,3_승인매출정보_train
